# Verbs only: LoRA rank sweep
Same lemma split as `01_p1_balanced_r4.ipynb` (same seed), verbs only, ranks 1-64. Safe to re-run after a disconnect: finished runs are skipped.

In [ ]:
%pip install -q --upgrade torchao
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os, json, tempfile
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

# ---------------- settings ----------------
LANGS = ['eng', 'spa', 'swe', 'fin']
SWEEP_LANGS = LANGS                        # e.g. ['eng'] to start with one language
FILES = {'fin': ['fin.1', 'fin.2']}
RANKS = [1, 2, 4, 8, 16, 32, 64]
N_TRAIN, N_EVAL = 10_000, 1000             # verb rows: train / val and test
EPOCHS, SEED = 6, 42
MODEL, MAXLEN, BS, LR = 'google/byt5-small', 64, 32, 3e-3
OUT = '/content/drive/MyDrive/lora/definitive' if IN_COLAB else '../data/definitive'
os.makedirs(f'{OUT}/sweep', exist_ok=True)

# ---------------- data: same lemma split as the Part 1 notebook (same seed), verbs only ----------------
def prepare(lang):
    url = 'https://raw.githubusercontent.com/unimorph/{}/master/{}'
    df = pd.concat([pd.read_csv(url.format(lang, f), sep='\t', header=None, names=['lemma', 'target', 'features'],
                                quoting=3, keep_default_na=False, dtype=str, on_bad_lines='skip')
                    for f in FILES.get(lang, [lang])]).apply(lambda c: c.str.strip())
    df = df[(df != '').all(axis=1) & ~df.target.str.contains('—', regex=False)]
    if lang == 'eng':                                # Wiktionary markup instead of forms
        df = df[~(df.target.isin(['countable', 'uncountable', 'comparable', 'comparative', 'superlative']) & (df.target != df.lemma))]
    df['pos'] = df.features.str.split(';').str[0].str.split('.').str[0]    # V.PTCP -> V
    df = df[df.pos.isin(['N', 'ADJ', 'V'])].drop_duplicates()
    # one row per (lemma, features); all attested forms kept in 'valid' for multi-valid scoring
    df = df.groupby(['lemma', 'features', 'pos'], as_index=False).target.agg(list).rename(columns={'target': 'valid'})
    df['target'] = df.valid.str[0]
    # 80/10/10 split by lemma (no lemma in two splits, also across POS)
    lem = df.lemma.drop_duplicates().sample(frac=1, random_state=SEED).tolist()
    a, b = int(.8 * len(lem)), int(.9 * len(lem))
    df['split'] = df.lemma.map({l: s for s, ls in zip(['train', 'val', 'test'], [lem[:a], lem[a:b], lem[b:]]) for l in ls})
    v = df[df.pos == 'V']
    out = {s: v[v.split == s].sample(min(n, (v.split == s).sum()), random_state=SEED).reset_index(drop=True)
           for s, n in [('train', N_TRAIN), ('val', N_EVAL), ('test', N_EVAL)]}
    print(lang, {s: (len(d), d.lemma.nunique()) for s, d in out.items()}, '(rows, lemmas)')
    return out

# ---------------- training / evaluation ----------------
tok = AutoTokenizer.from_pretrained(MODEL)
src = lambda d: (d.features + ' ' + d.lemma).tolist()      # Ale's input_v2_features_first

def train_eval(train, val, test, rank, save=None):
    def enc(d):
        x = dict(tok(src(d), max_length=MAXLEN, truncation=True))
        x['labels'] = tok(text_target=d.target.tolist(), max_length=MAXLEN, truncation=True)['input_ids']
        return Dataset.from_dict(x)

    def metric(ep):                                  # multi-valid exact match on val, used to pick the best epoch
        p = ep.predictions[0] if isinstance(ep.predictions, tuple) else ep.predictions
        p = tok.batch_decode(np.where(p >= 0, p, tok.pad_token_id), skip_special_tokens=True)
        return {'exact_match': float(np.mean([a.strip() in v for a, v in zip(p, val.valid)]))}

    model = get_peft_model(AutoModelForSeq2SeqLM.from_pretrained(MODEL), LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM, r=rank, lora_alpha=2 * rank, lora_dropout=0.1, bias='none',
        target_modules=['q', 'k', 'v', 'o', 'wi_0', 'wi_1', 'wo']))
    args = Seq2SeqTrainingArguments(
        output_dir=tempfile.mkdtemp(), eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='exact_match', greater_is_better=True,
        learning_rate=LR, per_device_train_batch_size=BS, per_device_eval_batch_size=BS * 2, num_train_epochs=EPOCHS,
        seed=SEED, predict_with_generate=True, generation_max_length=MAXLEN,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(), report_to='none', logging_steps=100)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=enc(train), eval_dataset=enc(val),
                             data_collator=DataCollatorForSeq2Seq(tok, model=model, pad_to_multiple_of=8),
                             compute_metrics=metric, processing_class=tok)
    trainer.train()
    hist = [(h['epoch'], h['eval_exact_match']) for h in trainer.state.log_history if 'eval_exact_match' in h]

    model.eval(); texts, preds = src(test), []
    for i in range(0, len(texts), 64):
        b = tok(texts[i:i + 64], return_tensors='pt', padding=True, max_length=MAXLEN, truncation=True).to(model.device)
        with torch.no_grad():
            preds += tok.batch_decode(model.generate(**b, max_length=MAXLEN), skip_special_tokens=True)
    if save: model.save_pretrained(save)
    out = test.assign(pred=[p.strip() for p in preds])
    out['ok'] = [p in v for p, v in zip(out.pred, out.valid)]      # multi-valid exact match
    out['strict'] = out.pred == out.target                          # exact match with the first listed form
    del model, trainer; torch.cuda.empty_cache()
    return out, hist

# ---------------- sweep: one result per run; the .json is written last, so it marks a finished run (skipped on re-run) ----------------
path = lambda l, r: f'{OUT}/sweep/{l}_r{r}.json'
for l in SWEEP_LANGS:
    todo = [r for r in RANKS if not os.path.exists(path(l, r))]
    if todo: d = prepare(l)
    for r in todo:
        out, _ = train_eval(d['train'], d['val'], d['test'], r)
        out.to_parquet(f'{OUT}/sweep/{l}_r{r}.parquet')     # test predictions of this run
        json.dump({'lang': l, 'rank': r, 'em': float(out.ok.mean()), 'strict': float(out.strict.mean())}, open(path(l, r), 'w'))
        print(f'{l} r={r}: multi-valid {out.ok.mean():.3f} | strict {out.strict.mean():.3f}')

tab = pd.DataFrame([json.load(open(path(l, r))) for l in SWEEP_LANGS for r in RANKS])
for c in ['em', 'strict']:
    print(f'\nVerbs, test accuracy by rank ({"multi-valid" if c == "em" else "strict"}):')
    print(tab.pivot(index='rank', columns='lang', values=c).round(3))

# ---------------- plots ----------------
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, c, t in zip(ax, ['em', 'strict'], ['multi-valid', 'strict']):
    for l, g in tab.groupby('lang'): a.plot(g['rank'], g[c], marker='o', label=l)
    a.set(xscale='log', xticks=RANKS, xticklabels=RANKS, xlabel='LoRA rank', ylabel='test accuracy',
          title=f'Verbs only: accuracy vs rank ({t})'); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/sweep_verbs.png', dpi=150); plt.show()